# Chapter 8: Tree-Based Algorithms and Ensemble Methods
## 📌 Summary
Decision Trees adalah model **non-parametric** yang membuat aturan if-else hierarkis. Ensemble methods (Random Forest, Gradient Boosting) menggabungkan banyak trees untuk performa jauh lebih baik.

## 🧠 Theoretical Deep-Dive

### 8.1 Decision Tree
Split berdasarkan impurity measure:
- **Gini impurity**: 1 - Σpᵢ² (default untuk classification)
- **Entropy**: -Σpᵢ·log₂(pᵢ)
- **MSE**: untuk regression

### 8.2 Bias-Variance Tradeoff
- Deep tree → low bias, high variance (overfitting)
- Shallow tree → high bias, low variance (underfitting)
- Hyperparameters: `max_depth`, `min_samples_split`, `min_samples_leaf`

### 8.3 Random Forest (Bagging)
- Train N trees pada **bootstrap samples** (random subset data)
- Setiap split hanya mempertimbangkan **random subset fitur** (√n_features)
- Prediksi = majority vote / average → variance reduction

### 8.4 Gradient Boosting
- Train trees **secara sequential**: setiap tree belajar dari **residual errors** tree sebelumnya
- Update: Fₘ(x) = Fₘ₋₁(x) + η·hₘ(x)
- η = learning rate; hₘ = weak learner (shallow tree)
- Bias reduction melalui boosting

### 8.5 AdaBoost
Memberikan **bobot lebih tinggi** pada misclassified samples sehingga tree berikutnya fokus pada kasus sulit.

## 💻 Code Reproduction

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

print("Train accuracy:", dt.score(X_train, y_train).round(4))
print("Test accuracy:", dt.score(X_test, y_test).round(4))
print("Feature importances:", dict(zip(load_iris().feature_names, dt.feature_importances_.round(3))))

plt.figure(figsize=(14, 6))
plot_tree(dt, feature_names=load_iris().feature_names, class_names=load_iris().target_names, filled=True, fontsize=9)
plt.title("Decision Tree (max_depth=3)")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Effect of max_depth
depths = range(1, 15)
train_scores, test_scores = [], []
for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_scores, label="Train", marker="o")
plt.plot(depths, test_scores, label="Test", marker="s")
plt.xlabel("max_depth"); plt.ylabel("Accuracy")
plt.title("Decision Tree: Depth vs Accuracy")
plt.legend(); plt.grid(); plt.tight_layout(); plt.show()
print("Best test depth:", depths[np.argmax(test_scores)], "acc:", max(test_scores).round(4))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print("Train accuracy:", rf.score(X_train, y_train))
print("Test accuracy:", rf.score(X_test, y_test).round(4))
print("\nClassification Report:")
print(classification_report(y_test, rf.predict(X_test)))

# Feature Importance
feature_names = load_breast_cancer().feature_names
idx = np.argsort(rf.feature_importances_)[::-1][:10]

plt.figure(figsize=(10, 4))
plt.bar(range(10), rf.feature_importances_[idx])
plt.xticks(range(10), [feature_names[i] for i in idx], rotation=45, ha="right")
plt.title("Random Forest: Top 10 Feature Importances")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

y_proba = gb.predict_proba(X_test)[:, 1]
print("Test accuracy:", gb.score(X_test, y_test).round(4))
print("ROC-AUC:", roc_auc_score(y_test, y_proba).round(4))

# Staged predictions (training curve)
train_scores = list(gb.staged_score(X_train, y_train))
test_scores = list(gb.staged_score(X_test, y_test))

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(train_scores, label="Train")
plt.plot(test_scores, label="Test")
plt.xlabel("n_estimators"); plt.ylabel("Accuracy")
plt.title("Gradient Boosting: Learning Curve")
plt.legend(); plt.grid(); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, VotingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# AdaBoost
ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42)
ada.fit(X_train, y_train)
print("AdaBoost acc:", ada.score(X_test, y_test).round(4))

# Voting Classifier
voting = VotingClassifier([
    ("lr", LogisticRegression(max_iter=1000)),
    ("dt", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("svc", SVC(probability=True))
], voting="soft")
voting.fit(X_train_s, y_train)
print("VotingClassifier acc:", voting.score(X_test_s, y_test).round(4))

# Bagging
bag = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42)
bag.fit(X_train, y_train)
print("BaggingClassifier acc:", bag.score(X_test, y_test).round(4))

## ✅ Chapter Summary
- **Decision Tree**: interpretable tapi mudah overfit → gunakan `max_depth`
- **Random Forest**: bagging + random feature subset → robust, sedikit tuning
- **Gradient Boosting**: sequential correction residuals → powerful tapi butuh tuning
- **AdaBoost**: reweight misclassified samples → baik untuk weak learners
- **Feature Importance** dari tree methods berguna untuk feature selection
- Untuk production: Random Forest = default; Gradient Boosting = best performance dengan tuning